# UC3 — Mobile Phone Screen (Glass): 1. Dataset Preparation

Before training or generation, the Phone Screen data must be fetched and arranged
into the layout Cosmos AnomalyGen expects. This notebook does exactly that.

> **How commands run in this tutorial.** All pipeline steps run inside the
> `cosmos-predict2` conda environment. In a notebook cell we prefix shell
> commands with `conda run -n cosmos-predict2` (add `--live-stream` to stream
> logs live). If you prefer, open a JupyterLab **Terminal**, run
> `conda activate cosmos-predict2` once, and paste the same commands without
> the `conda run` prefix.
>
> If that environment does not exist yet, build it first with the top-level
> [tutorial/notebooks/0-setup-cuda128.ipynb](../../0-setup-cuda128.ipynb) — see
> the prerequisite note below.

## 1.0 Set the project root

In [ ]:
# Resolve the repository root (the folder containing pyproject.toml) and cd into it,
# so every relative path below (datasets/, checkpoints/, results/, scripts/) resolves.
import os
d = os.getcwd()
while d != "/" and not os.path.exists(os.path.join(d, "pyproject.toml")):
    d = os.path.dirname(d)
LOCAL_PROJECT_DIR = d
os.chdir(LOCAL_PROJECT_DIR)
# Pipeline scripts read the finetuned models & write outputs under the repo root.
os.environ.setdefault("IMAGINAIRE_OUTPUT_ROOT", "./results")
print("Project root:", LOCAL_PROJECT_DIR)

## 1.1 Where the data comes from

| UC | Subject | Anomaly types | HF provides | Local prep |
|---|---|---|---|---|
| UC3 | Mobile phone screen | `Phone+oil`, `Phone+scratch`, `Phone+stain` | Masks + `defect_spec.jsonl` on Hugging Face | Manual Roboflow image download + `prepare_dataset_uc3.py` |

> **License notice.** You are responsible for confirming each dataset's license is
> fit for your intended use. See `datasets/README.md`.

## 1.2 Download & organize the dataset

UC3 images come from two sources:

- **Masks + `defect_spec.jsonl`** — the Hugging Face dataset repo
  [`nvidia/Cosmos-AnomalyGen-Glass-Masks`](https://huggingface.co/datasets/nvidia/Cosmos-AnomalyGen-Glass-Masks). Fetched automatically.
- **Anomaly + clean images** — a Roboflow dataset that **requires a manual browser
  download** (Roboflow does not allow unauthenticated programmatic download).

### Step 1 — Download the Roboflow zip (manual, browser required)

Roboflow does not allow unauthenticated programmatic download, so this step is manual
(full instructions also in
[`datasets/UC3_dataset_download_instructions.pdf`](../../../../datasets/UC3_dataset_download_instructions.pdf)):

1. Go to `https://universe.roboflow.com/vu-thi-thu-huyen/mobile-screen` (register / sign in).
2. Click **Use this Dataset** → **Fork Dataset** to copy it into your workspace.
3. Open your forked project → **Dataset** tab → **Export** (top-right).
4. Choose **Analyze or experiment** → continue → **Download anyway** when prompted.
5. Select **ZIP file** as the export format and download it.
6. Copy the zip onto this machine and note its path.

Screenshots of the Roboflow export (from NVIDIA's Physical AI Data Factory glass guide):

| Step | Screenshot |
|---|---|
| **Fork Dataset** confirmation dialog | ![](assets/roboflow/step3.png) |
| Open the forked project → **Dataset** tab | ![](assets/roboflow/step4.png) |
| **Export** button (top-right) | ![](assets/roboflow/step5.png) |
| Choose **Analyze or experiment** | ![](assets/roboflow/step6.png) |
| **Download anyway** prompt | ![](assets/roboflow/step7.png) |
| Select **ZIP file** format | ![](assets/roboflow/step8.png) |
| **Download** button (bottom of page) | ![](assets/roboflow/step9.png) |

### Step 2 — Run the preparation script

Pass the zip via `--zip` and pull masks/defect_spec from HF with `--masks-from-hf`:

In [ ]:
# Point this at your downloaded Roboflow zip. You can either edit the path here,
# or set a ROBOFLOW_ZIP environment variable before launching Jupyter.
import os
ROBOFLOW_ZIP = os.environ.get("ROBOFLOW_ZIP", "/path/to/mobile-screen.zip")
os.environ["ROBOFLOW_ZIP"] = ROBOFLOW_ZIP
print("Using ROBOFLOW_ZIP =", ROBOFLOW_ZIP)

In [ ]:
# Idempotent: skip if already prepared. Delete datasets/UC3_phone/ to force a fresh run.
!if [ -d datasets/UC3_phone/Phone/anomaly_image/oil ] && [ -f datasets/UC3_phone/defect_spec.jsonl ]; then \
   echo "datasets/UC3_phone already prepared - skipping (delete it to re-run)."; \
 else \
   conda run -n cosmos-predict2 python -m scripts.utilities.prepare_dataset_uc3 datasets/UC3_phone --zip "$ROBOFLOW_ZIP" --masks-from-hf; \
 fi

> **Masks only?** If you just want the masks + `defect_spec.jsonl` first (they come
> from HF and need no manual step), run without `--zip`:
> ```bash
> conda run -n cosmos-predict2 python -m scripts.utilities.prepare_dataset_uc3 datasets/UC3_phone --masks-from-hf
> ```
> The anomaly/clean images (and therefore generation) require the Roboflow zip.

## 1.3 Expected layout

After preparation, `datasets/UC3_phone/` looks like this:

```
datasets/UC3_phone/
  Phone/
    anomaly_image/<TYPE>/   real defect images
    mask/<TYPE>/            paired binary masks (<stem>_mask.png)
    clean_image/            defect-free canvases for generation
  defect_spec.jsonl         one line per defect: type + spatial_dependency
```

**Conventions the pipeline relies on:**
- `anomaly_types` are `[TEXTURE, TYPE]` pairs — the first element must match the texture folder name.
- Every anomaly image has a paired mask with the `_mask` suffix (`img_001.png` ↔ `img_001_mask.png`).
- `clean_image/` holds defect-free images used as canvases during generation.

Inspect it:

In [ ]:
!find datasets/UC3_phone -maxdepth 3 -type d | sort

## 1.4 The defect specification

`defect_spec.jsonl` tags each defect with a `spatial_dependency`
(`free` / `cad` / `text`) that controls how masks are placed during testcase
preparation (notebook 3).

In [ ]:
!cat datasets/UC3_phone/defect_spec.jsonl

## 1.5 Preview a few samples

Overlay the real masks on the real anomaly images to sanity-check the pairing.

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

dataset_dir = "datasets/UC3_phone"
pairs = []
for anom in sorted(glob.glob(f"{dataset_dir}/*/anomaly_image/*/*")):
    stem = os.path.splitext(os.path.basename(anom))[0]
    mask_dir = os.path.dirname(anom).replace("/anomaly_image/", "/mask/")
    # mask lookup is extension-agnostic and tolerates the "_mask" suffix or none
    cand = (glob.glob(os.path.join(mask_dir, stem + "_mask.*"))
            or glob.glob(os.path.join(mask_dir, stem + ".*")))
    if cand:
        pairs.append((anom, cand[0]))
    if len(pairs) >= 3:
        break

if not pairs:
    print("No (image, mask) pairs found under", dataset_dir, "- run the preparation step above first.")
else:
    fig, axes = plt.subplots(len(pairs), 3, figsize=(10, 3.2 * len(pairs)))
    axes = np.atleast_2d(axes)
    for r, (a, m) in enumerate(pairs):
        img = Image.open(a).convert("RGB")
        # masks may be authored at the original resolution; align to the image for overlay
        msk = Image.open(m).convert("L").resize(img.size)
        ov = np.array(img).copy(); mk = np.array(msk) > 127
        ov[mk] = (0.5 * ov[mk] + np.array([255, 0, 0]) * 0.5).astype("uint8")
        for ax, im, t in zip(axes[r], [img, msk, Image.fromarray(ov)],
                             ["anomaly image", "mask", "overlay"]):
            ax.imshow(im); ax.set_title(f"{os.path.basename(a).rsplit(chr(46), 1)[0]} — {t}", fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()

## Next Step

Proceed to [2-training.ipynb](./2-training.ipynb).